In [1]:
# Import the required libraries and utility functions
import pandas as pd

from src.utils import calculate_age, calculate_distance_km

In [2]:
# Load the training and test datasets
df_train = pd.read_csv('../data/fraudTrain.csv')
df_test = pd.read_csv('../data/fraudTest.csv')

# Confirm that the datasets were loaded successfully and display their dimensions
print("Datasets loaded successfully!")
print(f"Train dataset shape: {df_train.shape}")
print(f"Test dataset shape: {df_test.shape}")

Datasets loaded successfully!
Train dataset shape: (1296675, 23)
Test dataset shape: (555719, 23)


### Feature Selection & Drop Criteria

To prevent overfitting and optimize pipeline performance, non-predictive variables were removed based on the following technical criteria:

* **Unique Identifiers & PII (Unnamed: 0, cc_num, first, last, street, trans_num):** Lack reusable fraud patterns. Synthetic card numbers (cc_num) do not represent real BINs, and personal identity features force the model to memorize individuals rather than generalizable behaviors.
* **High-Cardinality Free Text (merchant, job, city, state):** Contain thousands of unstructured string values. Spatial signals are captured cleanly via calculated Haversine distance (distance_km), while business domain is preserved via category.
* **Temporal Redundancy (unix_time):** Raw numeric timestamp. Replaced by structured, human-readable temporal features extracted from trans_date_trans_time (e.g., trans_hour, trans_day).

In [3]:
# Define the list of columns to remove during the first preprocessing stage
cols_to_drop = [
    'Unnamed: 0', 'cc_num', 'first', 'last', 'street', 'trans_num', 'merchant', 'job', 'city', 'state', 'unix_time'
    ]

# Remove the selected columns from both training and test datasets
# Use errors='ignore' to prevent interruptions if a column has already been removed
df_train = df_train.drop(columns=cols_to_drop, errors='ignore')
df_test = df_test.drop(columns=cols_to_drop, errors='ignore')

# Display the updated dataset dimensions after feature removal
print(f"Train dataset shape: {df_train.shape}")
print(f"Test dataset shape: {df_test.shape}")

Train dataset shape: (1296675, 12)
Test dataset shape: (555719, 12)


In [4]:
# Convert the 'trans_date_trans_time' column from string to pandas datetime format
df_train['trans_date_trans_time'] = pd.to_datetime(df_train['trans_date_trans_time'])
df_train['dob'] = pd.to_datetime(df_train['dob'])

# Apply the same conversion to the test dataset
df_test['trans_date_trans_time'] = pd.to_datetime(df_test['trans_date_trans_time'])
df_test['dob'] = pd.to_datetime(df_test['dob'])

In [5]:
# Calculate the customer's age using the 'calculate_age' utility function
df_train["age"] = calculate_age(df_train)
df_test["age"] = calculate_age(df_test)

In [6]:
# Extract the transaction hour
df_train['trans_hour'] = df_train['trans_date_trans_time'].dt.hour
df_test['trans_hour'] = df_test['trans_date_trans_time'].dt.hour

# Extract the day of the week as a numeric value (0=Monday, 6=Sunday)
df_train['trans_day'] = df_train['trans_date_trans_time'].dt.dayofweek
df_test['trans_day'] = df_test['trans_date_trans_time'].dt.dayofweek

In [7]:
# Calculate the distance (km) between the customer and merchant locations using the 'calculate_distance_km' utility function
df_train['distance_km'] = calculate_distance_km(df_train)
df_test['distance_km'] = calculate_distance_km(df_test)

In [8]:
# Verify the resulting columns and data types
print(f'Train: {df_train.columns}')
print(f'\nTest: {df_test.columns}')
print(f"\n{df_train['trans_day'].dtype}")
print(f"\n{df_test['trans_day'].dtype}")

Train: Index(['trans_date_trans_time', 'category', 'amt', 'gender', 'zip', 'lat',
       'long', 'city_pop', 'dob', 'merch_lat', 'merch_long', 'is_fraud', 'age',
       'trans_hour', 'trans_day', 'distance_km'],
      dtype='str')

Test: Index(['trans_date_trans_time', 'category', 'amt', 'gender', 'zip', 'lat',
       'long', 'city_pop', 'dob', 'merch_lat', 'merch_long', 'is_fraud', 'age',
       'trans_hour', 'trans_day', 'distance_km'],
      dtype='str')

int32

int32


In [9]:
# Remove columns whose useful information has already been extracted
df_train = df_train.drop(
    columns=[
        'trans_date_trans_time', 'dob', 'lat', 'long',
        'merch_lat', 'merch_long', 'zip', 'city_pop'
    ],
    errors='ignore')

df_test = df_test.drop(
    columns=[
        'trans_date_trans_time', 'dob', 'lat', 'long',
        'merch_lat', 'merch_long', 'zip', 'city_pop'
    ],
    errors='ignore')

# Display the final feature set
print(f'Train: {df_train.columns}')
print(f'\nTest: {df_test.columns}')

Train: Index(['category', 'amt', 'gender', 'is_fraud', 'age', 'trans_hour',
       'trans_day', 'distance_km'],
      dtype='str')

Test: Index(['category', 'amt', 'gender', 'is_fraud', 'age', 'trans_hour',
       'trans_day', 'distance_km'],
      dtype='str')


In [10]:
# Encode the binary 'gender' feature (Female = 0, Male = 1)
df_train['gender'] = df_train['gender'].map({'F': 0, 'M': 1})
df_test['gender'] = df_test['gender'].map({'F': 0, 'M': 1})

# Apply One-Hot Encoding to the categorical 'category' feature
# Use drop_first=True to avoid perfect multicollinearity
df_train = pd.get_dummies(df_train, columns=['category'], drop_first=True)
df_test = pd.get_dummies(df_test, columns=['category'], drop_first=True)

In [11]:
# Verify that both datasets contain the same features in the same order
print(list(df_train.columns) == list(df_test.columns))

True


In [12]:
# Split the training dataset into features and target
X_train = df_train.drop(columns=['is_fraud'])
y_train = df_train['is_fraud']

# Split the test dataset into features and target
X_test = df_test.drop(columns=['is_fraud'])
y_test = df_test['is_fraud']

In [13]:
# Export the datasets as compressed Parquet files for efficient storage and I/O
X_train.to_parquet('../data/X_train.parquet', index=False)
X_test.to_parquet('../data/X_test.parquet', index=False)

y_train.to_frame().to_parquet('../data/y_train.parquet', index=False)
y_test.to_frame().to_parquet('../data/y_test.parquet', index=False)

In [14]:
# Display the final dataset dimensions
print("Filas y columnas de Train:", df_train.shape)
print("Filas y columnas de Test:", df_test.shape)

Filas y columnas de Train: (1296675, 20)
Filas y columnas de Test: (555719, 20)
